In [4]:
# Cell 2 — Setup
import os
import requests
import pandas as pd

NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", "270f34e4b2d24e42a34ef49fbfb006d2")  # <-- placeholder

if NEWSAPI_KEY == "270f34e4b2d24e42a34ef49fbfb006d2":
    print("⚠️ Mets ta clé dans la variable d'env NEWSAPI_KEY, ou remplace le placeholder ci-dessus.")
    # Tu peux décommenter la ligne suivante pour forcer l'erreur si tu préfères
    # raise ValueError("Missing NEWSAPI_KEY")

BASE_URL = "https://newsapi.org/v2"

def newsapi_get(endpoint: str, params: dict) -> dict:
    """
    endpoint: 'top-headlines' ou 'everything'
    params: dict de paramètres query
    """
    url = f"{BASE_URL}/{endpoint}"
    headers = {"X-Api-Key": NEWSAPI_KEY}
    r = requests.get(url, params=params, headers=headers, timeout=20)

    # NewsAPI renvoie généralement du JSON même en cas d'erreur
    try:
        data = r.json()
    except Exception:
        r.raise_for_status()
        raise RuntimeError("Réponse non-JSON inattendue")

    if r.status_code != 200 or data.get("status") != "ok":
        code = data.get("code")
        msg = data.get("message")
        raise RuntimeError(f"NewsAPI error (HTTP {r.status_code}) {code}: {msg}")

    return data

def articles_to_df(articles: list) -> pd.DataFrame:
    rows = []
    for a in articles:
        rows.append({
            "publishedAt": a.get("publishedAt"),
            "source": (a.get("source") or {}).get("name"),
            "title": a.get("title"),
            "description": a.get("description"),
            "url": a.get("url"),
        })
    return pd.DataFrame(rows)


⚠️ Mets ta clé dans la variable d'env NEWSAPI_KEY, ou remplace le placeholder ci-dessus.


In [5]:
# Cell 3 — Exemple 1 : récupérer 5 "top headlines" (France)
# -> pas besoin de thème, tu peux juste demander les gros titres d'un pays/catégorie
params = {
    "country": "us",
    "pageSize": 5,
    # "category": "technology",  # business | entertainment | general | health | science | sports | technology
    # "q": "IA",                 # optionnel: mots-clés
}

data = newsapi_get("top-headlines", params)
df = articles_to_df(data["articles"])
df


,publishedAt,source,title,description,url
0,2026-02-12T17:38:36Z,CNBC,Judge blocks Pete Hegseth's censure of Sen. Ma...,The order comes days after federal prosecutors...,https://www.cnbc.com/2026/02/12/kelly-hegseth-...
1,2026-02-12T17:38:02Z,Fox News,Nancy Guthrie disappearance: Investigators fin...,Investigators found a set of gloves amid the i...,https://foxnews.com/live-news/nancy-guthrie-di...
2,2026-02-12T17:22:34Z,NBC News,Jessie Diggins collapses in pain after securin...,Jessie Diggins of the United States battled th...,https://www.nbcnews.com/sports/olympics/jessie...
3,2026-02-12T16:00:00Z,MMA Fighting,"‘I don’t think he’s funny’: Zion Clark, wrestl...","Zion Clark, Valter Walker will compete at Kara...",https://www.mmafighting.com/latest-news/471238...
4,2026-02-12T15:00:15Z,CNBC,"January homes sales tank more than 8%, as Real...",Home sales in January fell more than expected ...,https://www.cnbc.com/2026/02/12/january-homes-...


In [7]:
print(df.loc[0, "description"])

The order comes days after federal prosecutors in Washington, D.C., tried and failed to get a grand jury to indict Sens. Mark Kelly and Elissa Slotkin.


In [6]:
data = newsapi_get("top-headlines", params)

print("status:", data.get("status"))
print("totalResults:", data.get("totalResults"))
print("articles len:", len(data.get("articles", [])))

# Si tu veux voir un extrait
print("first article keys:", (data["articles"][0].keys() if data.get("articles") else "NO_ARTICLES"))


status: ok
totalResults: 34
articles len: 5
first article keys: dict_keys(['source', 'author', 'title', 'description', 'url', 'urlToImage', 'publishedAt', 'content'])
